# Modul B · Kapitel 2.2 — Keyword Search

## Challenge: Finden, was wörtlich dasteht


**Lernziel:** Du kannst Term Frequency, Inverse Document Frequency und BM25 selbst rechnen, kennst
ihre Fallstricke und weißt, wo eine Suche über Wortübereinstimmung an ihre Grenze kommt.

Dieses Notebook baut den Retrieval-Schritt der RAG-Kette, in der Fassung ohne Embeddings:

```
Dokumente ──► Chunks ──► Embeddings ──► Vector Database ──► Retrieval ──► Prompt ──► Antwort
                                                             └─ hier ─┘
```

Gesucht wird zuerst in vier Dokumenten, die absichtlich schwierig sind, danach in den 90 Chunks
der Wissensbasis.

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |

**Wichtig:** Führe die Zellen **von oben nach unten** aus. Spätere Zellen brauchen die
Funktionen, die du vorher schreibst.

Es sind insgesamt **5 Challenges**.

---
## 0 · Setup

▶️ Führe die beiden nächsten Zellen aus.

Die erste holt die Pakete. Dieses Notebook rechnet **ohne Modell**: Jede Zahl kommt aus Python,
alle Ergebnisse sind deterministisch. Ein laufender Modellserver ist nicht nötig — auch nicht in
Google Colab.

Die Verbindung zum Modell steht für diesen Ordner an genau einer Stelle, in `helfer.py`. Wer
einen anderen Provider benutzt, ändert sie dort und nirgends sonst:

```python
BASIS_URL = "http://localhost:11434/v1"
API_KEY = "ollama"                # Ollama prüft den Key nicht
MODELL = "qwen3.5:0.8b"           # 1,0 GB, läuft auf jedem Laptop
EMBEDDING_MODELL = "nomic-embed-text"
```

In [ ]:
# ▶️ Pakete
import json
import math
import re
import sys
from collections import Counter
from pathlib import Path

try:
    from rank_bm25 import BM25Okapi
except ImportError:
    %pip install -q rank_bm25
    from rank_bm25 import BM25Okapi

import matplotlib.pyplot as plt

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})

print("Setup fertig ✔")

▶️ Die zweite Zelle lädt `helfer.py` und die vier Dokumente aus `daten/fallstricke/`.

In [ ]:
# ▶️ helfer.py finden, die vier Fallstrick-Dokumente laden
for kandidat in [Path.cwd(), *Path.cwd().parents]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

import helfer

dokumente = helfer.lade_dokumente(helfer.FALLSTRICKE)

print(f"{len(dokumente)} Dokumente aus daten/{helfer.FALLSTRICKE.name}/")
print()
for d in dokumente:
    print(f"  {d['id']:<34} {d['titel']}")

---
## 1 · Vier Dokumente, eine Frage

📖 Eine Suche wird nicht an leichten Fällen gemessen. Die vier Dokumente in `daten/fallstricke/`
handeln alle von Token, und nur eines beantwortet die Frage.

| Dokument | Was es ist | Was daran schwierig ist |
|---|---|---|
| Betriebshandbuch Rechenzentrum | Regelbetrieb einer Liegenschaft | sehr lang, erwähnt Token beiläufig |
| Sammelseite Token | ungepflegte Wiki-Seite | Keyword Stuffing: dieselben Wörter immer wieder |
| Runbook Zugriffstoken erneuern | die Anleitung mit der Antwort | benutzt andere Wörter als die Frage |
| Notiz Token und Kosten | Tokenizer und Kosten bei Sprachmodellen | dieselben Wörter, andere Bedeutung |

Die Suchanfrage lautet **„Token rotieren"**. Gemeint ist das Zugriffstoken eines technischen
Kontos: Es soll durch ein neues ersetzt werden, ohne dass die anbindende Anwendung ausfällt.

Die richtige Antwort steht im **Runbook**. Dort stehen die sechs Schritte, die Fristen und der
Notfallweg. Die Sammelseite wiederholt nur, dass man Token rotieren soll. Das Betriebshandbuch
verweist auf ein Runbook, ohne es zu beschreiben. Und in der Notiz bedeutet *Token* etwas
anderes: dort ist ein Token ein Textstück, kein Zugangsmerkmal.

▶️ Erst die Zahlen: Wie lang sind die Dokumente, und wie oft stehen die Suchwörter darin?

In [ ]:
# ▶️ Länge und Vorkommen der Suchwörter je Dokument
SUCHANFRAGE = "Token rotieren"
ZIEL_DOK = "03_runbook-token-rotation"      # das Dokument mit der Antwort

kopf = f"{'Dokument':<36}{'Zeichen':>9}{'Wörter':>8}{'token…':>8}{'rotier…':>9}"
print(kopf)
print("-" * len(kopf))

for d in dokumente:
    klein = d["text"].lower()
    marke = "  ← die Antwort" if d["id"] == ZIEL_DOK else ""
    zeile = (f"{d['id']:<36}{len(d['text']):>9,}{len(d['text'].split()):>8,}"
             f"{klein.count('token'):>8}{len(re.findall(r'rotier[a-zäöüß]*', klein)):>9}")
    print(zeile.replace(",", ".") + marke)

📖 Drei Zahlen aus der Tabelle.

Die **Sammelseite** schreibt *token* 62-mal auf 1.800 Zeichen, *rotieren* 22-mal. Sie ist das
kürzeste Dokument und hat die meisten Treffer.

Das **Betriebshandbuch** schreibt *token* siebenmal. Nicht, weil es um Token geht, sondern weil
es viermal so lang ist wie die anderen.

Das **Runbook** enthält *rotieren* kein einziges Mal. Es heißt dort durchgehend *erneuern*, und
die einzige Stelle mit derselben Wortwurzel ist die Kennung `rotation-2026-06` in einem
Beispielbefehl. Das Dokument mit der Antwort teilt sich mit der Frage genau ein Wort: *Token*.

---
## 2 · Tokenisierung für die Suche

📖 Eine Suche über Wortübereinstimmung braucht zuerst eine Festlegung, was ein Wort ist. Vier
Entscheidungen:

* **Kleinschreibung.** `Token`, `token` und `TOKEN` sollen dasselbe Wort sein.
* **Satzzeichen weg.** `Token.` und `Token,` sind keine eigenen Wörter.
* **Kennungen zusammenhalten.** `CVE-2026-3224` und `api-token` sind je ein Wort. Wer am
  Bindestrich trennt, macht aus einer eindeutigen Kennung drei beliebige Zahlen.
* **Stoppwörter entfernen.** *der, die, das, und, in* stehen in jedem Dokument. Sie sind häufig
  und trennen nichts — häufig ist nicht dasselbe wie wichtig.

Der letzte Punkt hat einen zweiten Grund. Gleich wird durch das häufigste Wort eines Dokuments
geteilt. Ohne Stoppwortliste ist dieses Wort in fast jedem deutschen Text *die* oder *der*, und
jeder Suchbegriff steht im Verhältnis dazu.

▶️ Die Stoppwortliste ist fertig. Sie ist bewusst kurz: nur Wörter ohne eigene Bedeutung.

In [ ]:
# ▶️ Eine kleine deutsche Stoppwortliste
STOPPWOERTER = {
    "der", "die", "das", "des", "dem", "den", "ein", "eine", "einer", "eines", "einem", "einen",
    "und", "oder", "aber", "auch", "als", "wie", "was", "wer", "wo", "wann", "warum", "welche",
    "ist", "sind", "war", "waren", "wird", "werden", "wurde", "wurden", "hat", "haben", "kann",
    "in", "im", "an", "am", "auf", "aus", "bei", "mit", "nach", "von", "vom", "vor", "zu", "zum",
    "zur", "für", "über", "unter", "durch", "gegen", "ohne", "um", "bis", "seit", "je", "pro",
    "ich", "du", "er", "sie", "es", "wir", "ihr", "man", "sich", "dieser", "diese", "dieses",
    "nicht", "kein", "keine", "nur", "noch", "schon", "so", "dass", "wenn", "dann", "sein",
}

print(f"{len(STOPPWOERTER)} Stoppwörter")

### 🛠️ Challenge 1: Tokenisierung

Schreibe `tokenisiere(text, stoppwoerter=STOPPWOERTER)`. Die Funktion gibt eine **Liste von
Suchwörtern** zurück, in der Reihenfolge des Textes:

* alles kleingeschrieben,
* Satzzeichen und Leerraum sind weg,
* Wörter mit Bindestrich bleiben zusammen: `cve-2026-3224` ist ein Wort, nicht drei,
* Stoppwörter kommen nicht vor.

Ein Wort besteht aus Buchstaben, Ziffern und Umlauten. Das Grundmuster dafür ist
`r"[a-z0-9äöüß]+"` — es zerreißt allerdings noch jede Kennung am Bindestrich.

*Tipp: `re.findall(muster, text.lower())` liefert alle Fundstellen als Liste. Um den Bindestrich
zuzulassen, hängst du an das Grundmuster eine Wiederholung aus Bindestrich und Grundmuster an:
`(?:-…)*`.*

In [ ]:
def tokenisiere(text, stoppwoerter=STOPPWOERTER):
    """Zerlegt einen Text in kleingeschriebene Suchwörter ohne Stoppwörter."""
    # TODO 1: Muster so erweitern, dass `cve-2026-3224` ein einziges Wort bleibt
    muster = r"[a-z0-9äöüß]+"

    # TODO 2: den kleingeschriebenen Text in Wörter zerlegen
    woerter = ...

    # TODO 3: Stoppwörter herausfiltern
    return ...


In [ ]:
# ✅ Selbsttest
assert tokenisiere("Das Token wird erneuert.") == ["token", "erneuert"], \
    "Kleinschreibung, Satzzeichen weg, Stoppwörter weg"
assert tokenisiere("Token, Token; Token!") == ["token", "token", "token"], \
    "Satzzeichen sind keine Wortbestandteile"
assert tokenisiere("Patch für CVE-2026-3224 einspielen") == ["patch", "cve-2026-3224", "einspielen"], \
    "Die Kennung muss ein Wort bleiben"
assert "api-token" in tokenisiere("Ein API-Token liegt im Secret-Store"), \
    "Auch api-token ist ein Wort"
assert tokenisiere("Die und der in im") == [], "Nur Stoppwörter ergeben eine leere Liste"
assert tokenisiere("") == [], "Leerer Text ergibt eine leere Liste"

print("✅ Challenge 1 gelöst")
print(tokenisiere("Wie oft muss das Zugriffstoken für svc-assethub erneuert werden?"))

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def tokenisiere(text, stoppwoerter=STOPPWOERTER):
    """Zerlegt einen Text in kleingeschriebene Suchwörter ohne Stoppwörter."""
    muster = r"[a-z0-9äöüß]+(?:-[a-z0-9äöüß]+)*"
    woerter = re.findall(muster, text.lower())
    return [w for w in woerter if w not in stoppwoerter]
```

`(?:…)` ist eine Gruppe ohne Rückgabewert. Ohne das `?:` würde `re.findall` nur den Inhalt der
Gruppe liefern, also den letzten Bindestrich-Teil statt des ganzen Wortes.

</details>

In [ ]:
# ▶️ Die vier Dokumente tokenisieren
korpus = [tokenisiere(d["text"]) for d in dokumente]
frage_tokens = tokenisiere(SUCHANFRAGE)

print(f"Suchanfrage {SUCHANFRAGE!r}  →  {frage_tokens}")
print()
kopf = f"{'Dokument':<36}{'Suchwörter':>12}   häufigste Wörter"
print(kopf)
print("-" * len(kopf))
for d, tokens in zip(dokumente, korpus):
    haeufig = ", ".join(f"{w} ({n})" for w, n in Counter(tokens).most_common(4))
    print(f"{d['id']:<36}{len(tokens):>12}   {haeufig}")

---
## 3 · Term Frequency

📖 Term Frequency misst, wie wichtig ein Wort **innerhalb eines Dokuments** ist. Zwei Schritte:

1. Vorkommen zählen: `#(t, D)`.
2. Durch das häufigste Wort desselben Dokuments teilen.

```
TF(t, D) = #(t, D) / max #(t′, D)
```

Der zweite Schritt ist die Normalisierung. Ohne sie gewinnt jedes lange Dokument allein dadurch,
dass es mehr Wörter enthält. Mit ihr liegt der Wert zwischen 0 und 1: Das häufigste Wort eines
Dokuments hat TF 1,0, ein Wort, das nicht vorkommt, hat TF 0.

Für eine Suchanfrage aus mehreren Wörtern werden die Einzelwerte addiert.

### 🛠️ Challenge 2: Term Frequency

Schreibe `term_frequency(term, tokens)` genau nach der Formel oben. `tokens` ist die Liste aus
`tokenisiere()`, Rückgabe ist eine Zahl zwischen 0 und 1.

Zwei Randfälle: eine leere Token-Liste ergibt `0.0`, ein Term, der nicht vorkommt, ebenfalls.

*Tipp: `Counter(tokens)` zählt alles auf einmal. `zaehler[term]` ist bei einem `Counter` auch
dann definiert, wenn der Term fehlt — dann ist er 0. Der Nenner ist `max(zaehler.values())`.*

In [ ]:
def term_frequency(term, tokens):
    """Relative Häufigkeit eines Terms, normalisiert auf das häufigste Wort des Dokuments."""
    if not tokens:
        return 0.0
    # TODO 1: alle Wörter zählen
    zaehler = ...
    # TODO 2: Vorkommen des Terms durch das häufigste Wort teilen
    return ...


In [ ]:
# ✅ Selbsttest
probe = tokenisiere("Token rotieren. Token prüfen. Token widerrufen. Rotieren hilft.")
assert probe.count("token") == 3 and probe.count("rotieren") == 2, "Die Probe hat 3× token, 2× rotieren"

assert term_frequency("token", probe) == 1.0, "Das häufigste Wort hat TF 1,0"
assert round(term_frequency("rotieren", probe), 3) == 0.667, "2 von 3 ergibt 0,667"
assert term_frequency("zugriffstoken", probe) == 0.0, "Ein Wort ohne Vorkommen hat TF 0"
assert term_frequency("token", []) == 0.0, "Leere Token-Liste ergibt 0.0"

print("✅ Challenge 2 gelöst")
for wort in ["token", "rotieren", "zugriffstoken"]:
    print(f"  TF({wort:<14}) = {term_frequency(wort, probe):.3f}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def term_frequency(term, tokens):
    """Relative Häufigkeit eines Terms, normalisiert auf das häufigste Wort des Dokuments."""
    if not tokens:
        return 0.0
    zaehler = Counter(tokens)
    return zaehler[term] / max(zaehler.values())
```

Die Abfrage auf die leere Liste steht vorne, weil `max()` auf einer leeren Folge einen
`ValueError` wirft.

</details>

In [ ]:
# ▶️ Term Frequency der Suchwörter in den vier Dokumenten
KURZ = ["Handbuch", "Sammelseite", "Runbook", "Notiz"]

kopf = f"{'Dokument':<14}{'häufigstes Wort':>18}" + "".join(f"{'TF ' + t:>14}" for t in frage_tokens) + f"{'Summe':>10}"
print(kopf)
print("-" * len(kopf))

tf_summen = []
for name, tokens in zip(KURZ, korpus):
    top_wort, top_zahl = Counter(tokens).most_common(1)[0]
    werte = [term_frequency(t, tokens) for t in frage_tokens]
    tf_summen.append(sum(werte))
    print(f"{name:<14}{top_wort + ' (' + str(top_zahl) + ')':>18}"
          + "".join(f"{w:>14.3f}" for w in werte) + f"{sum(werte):>10.3f}")

print()
print("Rangfolge nach Term Frequency:")
for rang, i in enumerate(sorted(range(4), key=lambda i: -tf_summen[i]), start=1):
    marke = "  ← die Antwort" if dokumente[i]["id"] == ZIEL_DOK else ""
    print(f"  {rang}. {KURZ[i]:<14}{tf_summen[i]:.3f}{marke}")

📖 Die Sammelseite steht vorn. *token* ist dort das häufigste Wort überhaupt (TF 1,0), *rotieren*
das zweithäufigste (TF 0,5). Genau das ist **Keyword Stuffing**: Ein Dokument wiederholt die
Wörter, nach denen gesucht wird. Term Frequency kann eine Wiederholung nicht von einer Erklärung
unterscheiden — sie zählt beides gleich.

Das Runbook liegt auf Platz 3, mit dem Wert, den ein einziges Suchwort hergibt. Das
Betriebshandbuch bildet das Schlusslicht, obwohl es *token* siebenmal enthält: Sein häufigstes
Wort ist `00`, aus Uhrzeiten wie `16:00`, und durch dessen zwölf Vorkommen wird geteilt. Auch das
gehört zur Tokenisierung — was als Wort gilt, entscheidet über den Nenner.

---
## 4 · Inverse Document Frequency

📖 Term Frequency sieht immer nur ein Dokument. Ob ein Wort überhaupt zwischen Dokumenten
unterscheidet, steht woanders: in der Zahl der Dokumente, die es enthalten — der **Document
Frequency** `df(t)`.

```
IDF(t) = log( N / df(t) )
```

`N` ist die Zahl der Dokumente im Speicher. Ein Wort, das in jedem Dokument steht, hat `df = N`
und damit IDF 0: Es trägt nichts zur Unterscheidung bei. Ein Wort in einem von vier Dokumenten
hat IDF `log 4 ≈ 1,386`.

Beide Teile zusammen ergeben die Gewichtung:

```
TF-IDF(t, D) = TF(t, D) · IDF(t)
```

Ein Wort zählt also nur dann viel, wenn es in **diesem** Dokument oft steht und in den **anderen**
selten.

### 🛠️ Challenge 3: Seltene Wörter stärker gewichten

Diese Challenge setzt zwei kleine Formeln zusammen:

1. inverse_document_frequency() misst: log(Anzahl Dokumente / Dokumente mit Term).
2. tf_idf() summiert für jeden Suchterm TF mal IDF.

Arbeite die drei TODO-Stellen der Reihe nach ab: zuerst df, dann IDF, dann die Summe.
Kommt ein Term nirgends vor, gib 0.0 zurück; so gibt es keine Division durch null.

Tipp: sum(1 for d in dokumente if term in d) zählt die Dokumente mit dem Term.


In [ ]:
def inverse_document_frequency(term, dokumente):
    """Wie selten ein Term über alle Dokumente hinweg ist: log(N / df)."""
    # TODO 1: in wie vielen Dokumenten kommt der Term vor?
    df = ...
    if df == 0:
        return 0.0
    # TODO 2: log(N / df)
    return ...


def tf_idf(terme, tokens, dokumente):
    """Summe aus TF mal IDF über alle Terme einer Suchanfrage."""
    # TODO 3: TF mal IDF über alle Terme aufsummieren
    raise NotImplementedError("Challenge 3: tf_idf() implementieren")


In [ ]:
# ✅ Selbsttest
A = tokenisiere("Token rotieren Token")
B = tokenisiere("Token Zutritt")
C = tokenisiere("Token Kälte")
mini = [A, B, C]

assert inverse_document_frequency("token", mini) == 0.0, "In allen drei Dokumenten: IDF 0"
assert round(inverse_document_frequency("rotieren", mini), 3) == round(math.log(3), 3), \
    "In einem von drei Dokumenten: log(3)"
assert inverse_document_frequency("pizza", mini) == 0.0, "Kein Vorkommen: IDF 0, keine Division durch null"

assert tf_idf(["token"], A, mini) == 0.0, "Ein Wort in jedem Dokument trägt nichts bei"
assert tf_idf(["rotieren"], A, mini) > 0, "Ein seltenes Wort trägt bei"
assert tf_idf(["rotieren"], B, mini) == 0.0, "In B kommt rotieren nicht vor"

print("✅ Challenge 3 gelöst")
for wort in ["token", "rotieren"]:
    print(f"  IDF({wort:<9}) = {inverse_document_frequency(wort, mini):.3f}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def inverse_document_frequency(term, dokumente):
    """Wie selten ein Term über alle Dokumente hinweg ist: log(N / df)."""
    df = sum(1 for tokens in dokumente if term in tokens)
    if df == 0:
        return 0.0
    return math.log(len(dokumente) / df)


def tf_idf(terme, tokens, dokumente):
    """Summe aus TF mal IDF über alle Terme einer Suchanfrage."""
    return sum(term_frequency(t, tokens) * inverse_document_frequency(t, dokumente)
               for t in terme)
```

`term in tokens` prüft auf der Liste linear. Bei großen Korpora wird stattdessen einmal ein
Index aufgebaut: ein Dict von Term auf die Menge der Dokumente, in denen er steht.

</details>

In [ ]:
# ▶️ IDF der Suchwörter und die neue Rangfolge
print("Suchwort      df   IDF")
for t in frage_tokens:
    df = sum(1 for tokens in korpus if t in tokens)
    print(f"{t:<12}{df:>4}   {inverse_document_frequency(t, korpus):.3f}")

print()
kopf = f"{'Dokument':<14}" + "".join(f"{'TF·IDF ' + t:>18}" for t in frage_tokens) + f"{'Summe':>10}"
print(kopf)
print("-" * len(kopf))

tfidf_summen = []
for name, tokens in zip(KURZ, korpus):
    werte = [term_frequency(t, tokens) * inverse_document_frequency(t, korpus) for t in frage_tokens]
    tfidf_summen.append(tf_idf(frage_tokens, tokens, korpus))
    print(f"{name:<14}" + "".join(f"{w:>18.3f}" for w in werte) + f"{sum(werte):>10.3f}")

print()
print("Rangfolge nach TF-IDF:")
for rang, i in enumerate(sorted(range(4), key=lambda i: -tfidf_summen[i]), start=1):
    marke = "  ← die Antwort" if dokumente[i]["id"] == ZIEL_DOK else ""
    print(f"  {rang}. {KURZ[i]:<14}{tfidf_summen[i]:.3f}{marke}")

📖 **Was sich ändert.** *token* verschwindet aus der Rechnung. Das Wort steht in allen vier
Dokumenten, `df = 4 = N`, IDF 0. Es entscheidet nichts mehr — dieselbe Eigenschaft, die
Stoppwörter unbrauchbar macht, trifft hier ein Fachwort. Übrig bleibt *rotieren* mit
IDF `log(4/3) ≈ 0,288`.

**Was sich nicht ändert.** Die Sammelseite steht weiter auf Platz 1. Sie häuft nicht nur das
häufige Wort, sondern auch das seltene. Gegen Keyword Stuffing ist IDF kein Mittel.

Auf Platz 2 bleibt die Notiz über Sprachmodelle. Sie benutzt dieselben Wörter wie die Frage und
meint etwas anderes: Ein Token ist dort ein Textstück, und *Rotation* beschreibt das Wechseln von
Log-Dateien. Eine Suche über Wortübereinstimmung vergleicht Zeichenketten, keine Bedeutungen —
diesen Unterschied kann sie nicht sehen.

Und das Runbook mit der Antwort fällt auf 0,000. Von seinen beiden Berührungspunkten mit der
Frage ist einer nichts mehr wert und der andere kommt nicht vor.

---
## 5 · BM25 aus dem Framework

📖 **BM25** — Best Match, Version 25 — ist die Ranking-Funktion, die in Suchmaschinen tatsächlich
läuft. Sie behält TF und IDF und ergänzt zwei Korrekturen.

**Sättigung.** Das zwanzigste Vorkommen eines Wortes sagt weniger als das zweite. BM25 lässt die
Termfrequenz deshalb nicht linear eingehen, sondern über eine Kurve, die sich einem Grenzwert
nähert. `k1` legt fest, wie schnell.

**Längennormalisierung.** Ein langes Dokument enthält jedes Wort öfter. BM25 teilt deshalb durch
die Dokumentlänge im Verhältnis zur durchschnittlichen Länge `avgdl`. `b` legt fest, wie stark.

```
                            tf · (k1 + 1)
BM25(t, D) = IDF(t) · ─────────────────────────────────────
                       tf + k1 · (1 − b + b · |D| / avgdl)
```

| Parameter | Wirkung | üblich |
|---|---|---|
| `k1` | klein: nur das Vorkommen zählt; groß: die Häufigkeit zählt fast linear | 1,2 bis 2,0 |
| `b` | `0`: Länge egal; `1`: volle Normalisierung auf die Durchschnittslänge | 0,75 |

`rank_bm25` bringt die Funktion fertig mit. `BM25Okapi` bekommt den Korpus als Liste von
Token-Listen, `get_scores()` die Suchanfrage in derselben Form.

▶️ Dieselbe Suchanfrage, dieselben vier Dokumente, jetzt mit den Standardparametern.

In [ ]:
# ▶️ BM25 mit den Standardparametern, im Vergleich zu TF und TF-IDF
bm25 = BM25Okapi(korpus, k1=1.5, b=0.75)
bm25_summen = [float(s) for s in bm25.get_scores(frage_tokens)]


def rang_von(werte):
    """Rang jedes Dokuments, 1 ist der beste."""
    ordnung = sorted(range(len(werte)), key=lambda i: -werte[i])
    return {i: rang for rang, i in enumerate(ordnung, start=1)}


spalten = [("TF", tf_summen), ("TF-IDF", tfidf_summen), ("BM25", bm25_summen)]
raenge = {name: rang_von(werte) for name, werte in spalten}

kopf = f"{'Dokument':<14}" + "".join(f"{name:>18}" for name, _ in spalten)
print(kopf)
print("-" * len(kopf))

for i, name in enumerate(KURZ):
    zeile = f"{name:<14}"
    for spalte, werte in spalten:
        zeile += f"{raenge[spalte][i]}. ({werte[i]:.3f})".rjust(18)
    if dokumente[i]["id"] == ZIEL_DOK:
        zeile += "  ← die Antwort"
    print(zeile)

print()
print(f"Vorsprung der Sammelseite vor der Notiz:"
      f"   TF-IDF Faktor {tfidf_summen[1] / tfidf_summen[3]:.1f}"
      f"   ·   BM25 Faktor {bm25_summen[1] / bm25_summen[3]:.1f}")

In [ ]:
# ▶️ Die drei Verfahren im Diagramm, je auf ihren besten Treffer normiert
verfahren = {"TF": tf_summen, "TF-IDF": tfidf_summen, "BM25": bm25_summen}
farben = [BLAU, ORANGE, TEAL]
breite = 0.24

plt.figure(figsize=(8, 4.5))
for i, ((name, werte), farbe) in enumerate(zip(verfahren.items(), farben)):
    hoehen = [w / max(werte) for w in werte]
    plt.bar([p + (i - 1) * 0.26 for p in range(len(KURZ))], hoehen, breite, label=name, color=farbe)

plt.xticks(range(len(KURZ)), ["Handbuch", "Sammelseite", "Runbook\n(die Antwort)", "Notiz"])
plt.ylabel("Score, normiert auf den besten Treffer")
plt.title("Alle drei Verfahren setzen die Sammelseite auf Platz 1")
plt.legend()
plt.show()

📖 BM25 dreht die Rangfolge nicht um. Es ändert die Abstände.

Nach TF-IDF liegt die Sammelseite um den Faktor 3,2 vor der Notiz, nach BM25 um den Faktor 1,2.
Die Sättigung nimmt den 22 Vorkommen von *rotieren* ihren Vorteil: Ab einem gewissen Punkt bringt
jedes weitere fast nichts mehr. Das ist die Antwort auf Keyword Stuffing.

Im Diagramm hat das Runbook keinen TF-IDF-Balken. Sein Wert ist 0,000, und ein Balken der Höhe
null ist nicht zu sehen.

Ein Hinweis zum kleinen Speicher: `rank_bm25` zieht die IDF nach unten hin auf einen Mindestwert.
Bei vier Dokumenten landen beide Suchwörter auf diesem Mindestwert, sodass hier fast nur
Termfrequenz und Länge entscheiden. Bei den 90 Chunks weiter unten greift die IDF wieder normal.

### 🛠️ Challenge 4: Die Parameter sichtbar machen

Schreibe `bm25_rangfolge(korpus, frage, k1=1.5, b=0.75)`. Eingabe sind der Korpus als Liste von
Token-Listen und die Suchanfrage als Token-Liste. Rückgabe ist eine nach Score **absteigend
sortierte Liste von Paaren** `(index, score)`; `index` ist die Stelle des Dokuments im Korpus.

*Tipp: `BM25Okapi(korpus, k1=…, b=…)` nimmt die beiden Parameter im Konstruktor entgegen.
`get_scores(frage)` liefert die Scores in der Reihenfolge des Korpus. `sorted(enumerate(scores),
key=lambda paar: -paar[1])` dreht das in eine Rangfolge um.*

In [ ]:
def bm25_rangfolge(korpus, frage, k1=1.5, b=0.75):
    """Rangfolge nach BM25 als Liste von (index, score), höchster Score zuerst."""
    # TODO 1: BM25Okapi mit k1 und b aufbauen
    bm25 = ...
    # TODO 2: Scores für die Suchanfrage holen
    scores = ...
    # TODO 3: absteigend nach Score sortiert zurückgeben
    return ...


In [ ]:
# ✅ Selbsttest
rangfolge = bm25_rangfolge(korpus, frage_tokens)

assert len(rangfolge) == 4, "Vier Dokumente, vier Plätze"
assert sorted(i for i, _ in rangfolge) == [0, 1, 2, 3], "Jedes Dokument kommt genau einmal vor"
assert all(rangfolge[i][1] >= rangfolge[i + 1][1] for i in range(3)), "Absteigend sortiert"
assert rangfolge[0][0] == 1, "Auch BM25 setzt die Sammelseite auf Platz 1"

ohne_laenge = dict(bm25_rangfolge(korpus, frage_tokens, b=0.0))
volle_laenge = dict(bm25_rangfolge(korpus, frage_tokens, b=1.0))
assert ohne_laenge[0] > volle_laenge[0], \
    "Ohne Längennormalisierung steht das lange Betriebshandbuch besser da"

flach = dict(bm25_rangfolge(korpus, frage_tokens, k1=0.1))
steil = dict(bm25_rangfolge(korpus, frage_tokens, k1=3.0))
assert flach[1] - flach[3] < steil[1] - steil[3], \
    "Kleines k1 sättigt früher, der Abstand der Sammelseite schrumpft"

print("✅ Challenge 4 gelöst")
for rang, (i, score) in enumerate(rangfolge, start=1):
    print(f"  {rang}. {KURZ[i]:<14}{score:.3f}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def bm25_rangfolge(korpus, frage, k1=1.5, b=0.75):
    """Rangfolge nach BM25 als Liste von (index, score), höchster Score zuerst."""
    bm25 = BM25Okapi(korpus, k1=k1, b=b)
    scores = bm25.get_scores(frage)
    return sorted(enumerate(float(s) for s in scores), key=lambda paar: (-paar[1], paar[0]))
```

Der zweite Teil des Sortierschlüssels, `paar[0]`, entscheidet bei gleichem Score nach der
Reihenfolge im Korpus. Ohne ihn hängt das Ergebnis bei Gleichstand von der Sortierstabilität ab.

</details>

In [ ]:
# ▶️ Das Parametergitter: Rang und Score je Dokument
kopf = f"{'k1':>5}{'b':>7}" + "".join(f"{n:>16}" for n in KURZ)
print(kopf)
print("-" * len(kopf))

for k1 in (0.5, 1.5, 3.0):
    for b in (0.0, 0.5, 0.75, 1.0):
        rangliste = bm25_rangfolge(korpus, frage_tokens, k1=k1, b=b)
        werte = dict(rangliste)
        platz = {i: rang for rang, (i, _) in enumerate(rangliste, start=1)}
        zeile = f"{k1:>5.1f}{b:>7.2f}"
        for i in range(len(KURZ)):
            zeile += f"{platz[i]}. ({werte[i]:.3f})".rjust(16)
        print(zeile)
    print()

In [ ]:
# ▶️ Derselbe Inhalt, doppelte Länge
handbuch = dokumente[0]["text"]
korpus_plus = korpus + [tokenisiere(handbuch + "\n\n" + handbuch)]
namen_plus = KURZ + ["Handbuch ×2"]

print(f"{'Dokument':<14}{'Suchwörter':>12}{'b = 0':>10}{'b = 0,75':>11}")
print("-" * 47)
ohne = dict(bm25_rangfolge(korpus_plus, frage_tokens, b=0.0))
mit = dict(bm25_rangfolge(korpus_plus, frage_tokens, b=0.75))
for i, name in enumerate(namen_plus):
    print(f"{name:<14}{len(korpus_plus[i]):>12}{ohne[i]:>10.3f}{mit[i]:>11.3f}")

print()
print(f"Aufschlag der verdoppelten Fassung:  b = 0    {ohne[4] / ohne[0] - 1:+.0%}")
print(f"                                     b = 0,75 {mit[4] / mit[0] - 1:+.0%}")

📖 Die letzte Tabelle ist ein kontrollierter Fall: Das Betriebshandbuch steht ein zweites Mal im
Speicher, mit sich selbst verdoppelt. Der Inhalt ist identisch, nur die Länge nicht. Ohne
Längennormalisierung bekommt die verdoppelte Fassung dafür einen um ein Fünftel höheren Score.
Mit `b = 0,75` bleibt fast nichts davon übrig.

Im Parametergitter zeigt sich dasselbe am echten Betriebshandbuch. Es ist mehr als doppelt so
lang wie der Durchschnitt der vier Dokumente. Bei `k1 = 3,0` und `b = 0` steht es vor dem
Runbook, bei `b = 0,5` dahinter — sein Vorsprung kommt aus dem Umfang, nicht aus dem Thema.

`k1` wirkt in die andere Richtung. Bei `k1 = 0,5` liegen Sammelseite und Notiz mit 0,545 gegen
0,504 fast gleichauf: Es zählt kaum noch, wie oft ein Wort vorkommt, sondern nur noch, **ob**.
Bei `k1 = 3,0` stehen 1,383 gegen 1,000 — derselbe Abstand ist neunmal so groß, weil die
Häufigkeit wieder voll durchschlägt.

Was keine Parameterwahl ändert: Das Runbook bleibt auf dem letzten Platz. Ein Wort, das im
Dokument nicht steht, lässt sich nicht gewichten.

---
## 6 · Von vier Dokumenten auf 90 Chunks

📖 Vier Dokumente sind ein Testfall, keine Knowledge Base. Die Wissensbasis des Security
Operations Center liegt vorgechunkt in `daten/chunks.json`: 90 Chunks aus elf Dokumenten. Dazu
gibt es zehn Fragen mit den Dokumenten, in denen die Antwort steht.

Für BM25 ändert sich dadurch nichts — ein Chunk ist eine kurze Textstelle mit einer Kennung, mehr
braucht das Verfahren nicht. Zwei Dinge ändern sich in den Zahlen: Der Speicher hat 90 statt vier
Einträge, sodass die IDF wieder unterscheidet. Und die Einträge sind ähnlich lang, sodass `b`
weniger ins Gewicht fällt.

Gemessen wird mit **Recall@k**: der Anteil der Fragen, bei denen unter den ersten `k` Treffern
mindestens ein Chunk aus einem erwarteten Dokument liegt.

In [ ]:
# ▶️ Chunks und Evaluationsfragen laden
chunks = helfer.lade_chunks()
fragen = helfer.lade_fragen()

laengen = [len(tokenisiere(c["text"])) for c in chunks]
print(f"{len(chunks)} Chunks aus {len({c['dok_id'] for c in chunks})} Dokumenten")
print(f"Suchwörter je Chunk: {min(laengen)} bis {max(laengen)}, im Mittel {sum(laengen) // len(laengen)}")
print(f"{len(fragen)} Evaluationsfragen")
print()
for f in fragen[:3]:
    print(f"  {f['frage']}")
    print(f"     erwartet: {', '.join(f['erwartete_dok_ids'])}")

### 🛠️ Challenge 5: Suche über die Chunks

Schreibe `bm25_suche(frage, chunks, n=5)`. Die Funktion tokenisiert alle Chunk-Texte, baut daraus
einen `BM25Okapi`-Index, wertet die Frage darauf aus und gibt die **`n` besten Chunks** zurück.

Jeder Treffer ist der Chunk selbst, ergänzt um sein Feld `score`. Damit kann
`helfer.zeige_treffer()` die Liste anzeigen:

```python
{"chunk_id": "cve-2026-3224#01", "dok_id": …, "titel": …, "text": …, "position": …, "score": 8.2}
```

*Tipp: `{**chunk, "score": float(s)}` erzeugt eine Kopie mit einem zusätzlichen Feld.
`sorted(range(len(chunks)), key=lambda i: -scores[i])[:n]` liefert die Positionen der besten
Treffer.*

In [ ]:
def bm25_suche(frage, chunks, n=5, k1=1.5, b=0.75):
    """Die n Chunks mit dem höchsten BM25-Score zu einer Frage."""
    # TODO 1: die Chunk-Texte tokenisieren und einen BM25Okapi-Index bauen
    korpus = ...
    bm25 = ...
    # TODO 2: die tokenisierte Frage auswerten
    scores = ...
    # TODO 3: die n besten Positionen bestimmen und die Chunks mit score zurückgeben
    raise NotImplementedError("Challenge 5: bm25_suche() implementieren")


In [ ]:
# ✅ Selbsttest
treffer = bm25_suche("Wie lange werden Firewall-Logs aufbewahrt?", chunks, n=5)

assert len(treffer) == 5, "Fünf Treffer erwartet"
assert all({"chunk_id", "dok_id", "text", "score"} <= set(t) for t in treffer), \
    "Jeder Treffer ist der Chunk plus das Feld score"
assert all(treffer[i]["score"] >= treffer[i + 1]["score"] for i in range(4)), "Absteigend sortiert"
assert len({t["chunk_id"] for t in treffer}) == 5, "Kein Chunk darf doppelt vorkommen"
assert any(t["dok_id"] == "policy-logging-und-aufbewahrung" for t in treffer), \
    "Die Richtlinie zur Aufbewahrung gehört unter die ersten fünf"
assert len(bm25_suche("Notfall-Patch", chunks, n=3)) == 3, "n steuert die Trefferzahl"

print("✅ Challenge 5 gelöst")
helfer.zeige_treffer(treffer)

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def bm25_suche(frage, chunks, n=5, k1=1.5, b=0.75):
    """Die n Chunks mit dem höchsten BM25-Score zu einer Frage."""
    korpus = [tokenisiere(c["text"]) for c in chunks]
    bm25 = BM25Okapi(korpus, k1=k1, b=b)
    scores = bm25.get_scores(tokenisiere(frage))
    beste = sorted(range(len(chunks)), key=lambda i: -scores[i])[:n]
    return [{**chunks[i], "score": float(scores[i])} for i in beste]
```

Der Index wird hier bei jedem Aufruf neu gebaut. Bei 90 Chunks fällt das nicht auf; bei
hunderttausend baut man ihn einmal und gibt ihn der Suchfunktion mit.

</details>

In [ ]:
# ▶️ Recall@k über alle zehn Fragen
def recall_at_k(fragen, chunks, k):
    """Anteil der Fragen mit mindestens einem Treffer aus einem erwarteten Dokument."""
    getroffen = 0
    for f in fragen:
        dok_ids = {t["dok_id"] for t in bm25_suche(f["frage"], chunks, n=k)}
        getroffen += any(d in dok_ids for d in f["erwartete_dok_ids"])
    return getroffen / len(fragen)


for k in (1, 3, 5, 10):
    quote = recall_at_k(fragen, chunks, k)
    print(f"Recall@{k:<3} {quote:5.0%}   ({round(quote * len(fragen))} von {len(fragen)} Fragen)")

In [ ]:
# ▶️ Die Fragen, bei denen der erste Treffer danebenliegt
for f in fragen:
    treffer = bm25_suche(f["frage"], chunks, n=5)
    if treffer[0]["dok_id"] in f["erwartete_dok_ids"]:
        continue

    print(f"❓ {f['frage']}")
    print(f"   erwartet: {', '.join(f['erwartete_dok_ids'])}")
    for rang, t in enumerate(treffer, start=1):
        marke = "✔" if t["dok_id"] in f["erwartete_dok_ids"] else " "
        print(f"   {marke} {rang}. {t['score']:5.2f}  {t['chunk_id']:<32} "
              f"{' '.join(t['text'].split())[:60]}…")
    print()

📖 **Recall@5 liegt bei 10 von 10 Fragen.** Unter den ersten fünf Chunks ist immer einer aus dem
erwarteten Dokument. Auf Platz 1 sind es 8 von 10.

Die beiden Fehlschläge auf Platz 1 zeigen dieselbe Ursache aus zwei Richtungen:

**„Welchen CVSS-Wert hat CVE-2026-3224?"** Ganz oben steht ein Chunk aus einem anderen Advisory,
weil dort das Wort *CVSS-Wert* wörtlich vorkommt. Das gesuchte Advisory schreibt an der
entscheidenden Stelle `CVSS v3.1 Base Score | 9.8 (critical)` — die Zeichenfolge *CVSS-Wert*
steht dort nirgends.

**„Wie schnell muss ein Notfall-Patch auf Servern eingespielt werden?"** Ganz oben steht ein
Chunk mit vierzehn Suchwörtern: die Referenzliste eines Advisories, in der *Notfall-Patch* als
Stichwort auftaucht. Kurz und voller Suchwörter — genau das, was BM25 hoch bewertet. Eine Antwort
steht darin nicht.

Ein Hinweis zur Kennzahl: Recall@5 zählt das **Dokument**, nicht die Stelle. Bei beiden Fragen
liegt zwar ein Chunk aus dem richtigen Dokument unter den ersten fünf, aber nicht der mit der
Antwort — die steht in `cve-2026-3224#01` und in `runbook-patch-management#01`. Eine strengere
Kennzahl würde auf Chunk-Ebene prüfen.

---
## 7 · Wo Keyword Search trägt und wo nicht

📖 Zehn von zehn Fragen mit einem Treffer unter den ersten fünf — das ist ein brauchbares
Ergebnis, und es hat einen Grund: Die Evaluationsfragen benutzen die Wörter der Dokumente.
*Firewall-Logs*, *Notfall-Patch*, *SEV-1*, *CVE-2026-3224* stehen wörtlich in der Wissensbasis.

**Darin ist Keyword Search stark:**

* exakte Kennungen — `CVE-2026-4410`, `INC-2026-0311`, `NP-SA-2026-014`,
* Produkt- und Systemnamen — `SentinelGrid`, `NorthPeak SecureGate`, `Vaultbird`,
* Konfigurationsschlüssel und Fehlercodes — `max_key_share_entries`, `adm-`.

Solche Zeichenfolgen haben keine Synonyme. Eine Kennung steht in wenigen Chunks oder in einem
einzigen; ihre IDF ist entsprechend hoch.

▶️ Drei Suchen nach Kennungen.

In [ ]:
# ▶️ Kennungen sind der beste Fall für Keyword Search
for kennung in ["CVE-2026-4410", "INC-2026-0311", "max_key_share_entries"]:
    treffer = bm25_suche(kennung, chunks, n=3)
    print(f"🔎 {kennung}")
    for rang, t in enumerate(treffer, start=1):
        print(f"   {rang}. {t['score']:6.2f}  {t['chunk_id']:<32} {t['titel'][:44]}")
    print()

📖 Der Score fällt nach dem ersten Treffer steil ab, teils auf null: Kein anderer Chunk enthält
die Zeichenfolge. Genau das macht die Suche über Wortübereinstimmung unverzichtbar — ein
Verfahren, das Bedeutung vergleicht, verwechselt zwei CVE-Kennungen mühelos.

Bei `INC-2026-0311` ist zugleich die Kehrseite der Längennormalisierung zu sehen: Ganz oben steht
die Referenzliste eines Advisories, die die Kennung einmal nennt. Erst dahinter kommt das
Post-Mortem, das den Vorfall beschreibt und die Kennung zweimal nennt. Der kürzere Chunk gewinnt
trotz der geringeren Zahl von Vorkommen.

**Und darin ist sie schwach:** wenn die Frage andere Wörter benutzt als das Dokument. Das ist der
Normalfall, sobald nicht mehr die Autorin der Dokumente die Fragen stellt.

▶️ Dieselben sechs Sachverhalte, mit anderen Wörtern gefragt.

In [ ]:
# ▶️ Umformulierte Fragen — dieselbe Wissensbasis
UMFORMULIERT = [
    ("Wie gefährlich ist die Lücke im VPN-Zugang von NorthPeak?", ["cve-2026-3224"]),
    ("Wie viel Zeit bleibt für ein dringendes Sicherheitsupdate?", ["runbook-patch-management"]),
    ("Wer stuft einen Vorfall als besonders schwerwiegend ein?", ["runbook-incident-response"]),
    ("Wie lange bleiben die Aufzeichnungen der Firewall gespeichert?", ["policy-logging-und-aufbewahrung"]),
    ("Was geschieht mit dem Zugang, wenn jemand die Firma verlässt?", ["runbook-zugriffskontrolle"]),
    ("Wie viel Last hält die SIEM-Plattform im Alltag aus?", ["systemdoku-sentinelgrid"]),
]

for frage, erwartet in UMFORMULIERT:
    treffer = bm25_suche(frage, chunks, n=5)
    dok_ids = [t["dok_id"] for t in treffer]
    marke = "✔" if dok_ids[0] in erwartet else "✗"
    print(f"{marke} {frage}")
    print(f"    Suchwörter: {tokenisiere(frage)}")
    print(f"    Platz 1: {treffer[0]['chunk_id']}  ({treffer[0]['score']:.2f})   erwartet: {erwartet[0]}")

vergleich = [
    ("Fragen aus daten/fragen.json", [(f["frage"], f["erwartete_dok_ids"]) for f in fragen]),
    ("dieselben Sachverhalte, umformuliert", UMFORMULIERT),
]

print()
print(f"{'Fragensatz':<38}{'Recall@1':>10}{'Recall@5':>10}")
print("-" * 58)
for name, satz in vergleich:
    quoten = []
    for k in (1, 5):
        getroffen = sum(1 for frage, erwartet in satz
                        if any(t["dok_id"] in erwartet for t in bm25_suche(frage, chunks, n=k)))
        quoten.append(getroffen / len(satz))
    print(f"{name:<38}{quoten[0]:>10.0%}{quoten[1]:>10.0%}")

📖 Auf Platz 1 bricht das Ergebnis ein. Die Suchwörter der umformulierten Fragen — *gefährlich*,
*Sicherheitsupdate*, *Aufzeichnungen*, *Last* — stehen so nicht in den Dokumenten. Dort heißt es
*CVSS*, *Patch*, *Logs*, *Ereignisse pro Sekunde*. Beide Seiten meinen dasselbe und teilen kein
Wort.

Damit sind die Grenzen benannt, und sie sind dieselben wie bei den vier Fallstrick-Dokumenten:

| Fall | Was passiert |
|---|---|
| Keyword Stuffing | Wiederholung sieht aus wie Relevanz; BM25 dämpft sie, hebt sie aber nicht auf |
| sehr großes Dokument | mehr Text bedeutet mehr Treffer; `b` normalisiert das |
| andere Wortwahl | kein gemeinsames Wort, kein Score — unabhängig von den Parametern |
| andere Bedeutung | dieselben Wörter, anderes Thema; für das Verfahren nicht unterscheidbar |

Die letzten beiden Zeilen lassen sich mit Zählen nicht lösen. Dafür braucht es eine Darstellung,
in der *Aufzeichnungen* und *Logs* nahe beieinander liegen und die beiden Bedeutungen von *Token*
weit auseinander — das ist die Aufgabe von Semantic Search.

▶️ Die letzte Zelle legt die BM25-Ergebnisse als Datei ab.

In [ ]:
# ▶️ Ergebnisse sichern
ergebnisse = {
    "verfahren": "bm25-okapi",
    "parameter": {"k1": 1.5, "b": 0.75},
    "fallstricke": {
        "suchanfrage": SUCHANFRAGE,
        "ziel_dok": ZIEL_DOK,
        "tf": {d["id"]: round(w, 4) for d, w in zip(dokumente, tf_summen)},
        "tf_idf": {d["id"]: round(w, 4) for d, w in zip(dokumente, tfidf_summen)},
        "bm25": {d["id"]: round(w, 4) for d, w in zip(dokumente, bm25_summen)},
    },
    "fragen": [
        {"frage": f["frage"],
         "erwartete_dok_ids": f["erwartete_dok_ids"],
         "treffer": [{"chunk_id": t["chunk_id"], "score": round(t["score"], 4)}
                     for t in bm25_suche(f["frage"], chunks, n=5)]}
        for f in fragen
    ],
}

pfad = helfer.DATEN / "02_keyword_treffer.json"
pfad.write_text(json.dumps(ergebnisse, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")

print(f"{pfad.name} geschrieben: {len(ergebnisse['fragen'])} Fragen mit je 5 Treffern")

---
## 8 · Was du gebaut hast

* `tokenisiere()` — Kleinschreibung, Satzzeichen weg, Kennungen zusammen, Stoppwörter weg.
* `term_frequency()` — Vorkommen, normalisiert auf das häufigste Wort des Dokuments.
* `inverse_document_frequency()` und `tf_idf()` — die Gewichtung über den ganzen Speicher hinweg.
* `bm25_rangfolge()` — BM25 mit `k1` und `b` als Stellschrauben, an vier Dokumenten gemessen.
* `bm25_suche()` — die Suche über 90 Chunks, mit Recall@k über zehn Fragen.

Die Zahlen zum Mitnehmen: Auf Fragen, die die Wörter der Dokumente benutzen, liefert BM25 bei
allen zehn Fragen einen Treffer unter den ersten fünf, bei acht von zehn schon auf Platz 1. Sind
dieselben Sachverhalte anders formuliert, bleibt auf Platz 1 einer von sechs übrig. Keyword
Search findet, was wörtlich dasteht — nicht, was gemeint ist.

---
### 🔬 Bonus — ohne Lösung

**1. Stemming und Kompositazerlegung.** Deutsche Wortformen sind der Hauptgrund für verpasste
Treffer: *erneuern*, *erneuert*, *Erneuerung* sind für die Suche drei Wörter, und
*Zugriffstoken* trifft *Token* nicht. Vorgehen:

1. `tokenisiere()` um einen Stemmer erweitern — etwa `SnowballStemmer("german")` aus `nltk` oder
   eine eigene Regel, die die häufigsten Endungen abschneidet.
2. Lange Komposita zusätzlich zerlegen: Steht ein bekanntes Wort am Ende eines längeren, beide
   Formen in die Token-Liste aufnehmen.
3. Recall@1 und Recall@5 über die zehn Fragen und über `UMFORMULIERT` erneut messen.

Zwei Fragen dazu: Steigt die Trefferquote — und findet die Suche jetzt das Runbook aus Abschnitt 1?

**2. Query Expansion mit dem Modell.** Statt die Dokumente anzupassen, wird die Frage erweitert.
Vorgehen:

1. Mit `helfer.frage_llm()` zu jeder Frage aus `UMFORMULIERT` fünf Suchwörter erzeugen lassen —
   Synonyme und Fachbegriffe, Antwortformat JSON.
2. Die erzeugten Wörter an die tokenisierte Frage anhängen und `bm25_suche()` erneut laufen lassen.
3. Recall@1 vergleichen.

Interessant ist dabei die Kehrseite: Wie oft holt die Erweiterung Chunks nach oben, die nichts
mit der Frage zu tun haben?